# Lab for Lesson 1 - Introduction to AI Methods for Embedded Systems (Using Keras)

## Importing Required Libraries

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
from keras import layers
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import random
import sys
import ctypes
import threading
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from pathlib import Path
import seaborn as sns
import librosa
import scipy.signal as signalpy
from scipy.io import wavfile
from ansys.scadeone.core import ScadeOne
from ansys.scadeone.core.svc.pywrapper.python_wrapper import PythonWrapper

## Utilities

In [ ]:
keras.utils.set_random_seed(42)
print("Keras version:", keras.__version__)
print("TensorFlow version:", tf.__version__)
print(tf.config.list_physical_devices())


## Introducing a regression problem: Neural-Network Sensor Calibration

### Generating the Sensor Dataset

In [ ]:
rng = np.random.default_rng(42)
n_samples = 2000
temperature_c = rng.uniform(
    low=-20.0,
    high=80.0,
    size=n_samples
)
ideal_voltage = (
    0.50
    + 0.012 * temperature_c
    + 0.00004 * temperature_c**2
)
measured_voltage = ideal_voltage + rng.normal(
    loc=0.0,
    scale=0.015,
    size=n_samples
)
# The parameter “-1” of the reshape method allows numpy to compute the dimension of that axis.
X = measured_voltage.reshape(-1, 1).astype("float32")
y = temperature_c.reshape(-1, 1).astype("float32")

In [ ]:
print(measured_voltage)

In [ ]:
print(X)

In [ ]:
print(temperature_c)

In [ ]:
print(y)

### Visualize the generated dataset

In [ ]:
plt.scatter(
    measured_voltage,
    temperature_c,
    s=10,
    alpha=0.5
)
plt.xlabel("Sensor voltage [V]")
plt.ylabel("Temperature [°C]")
plt.title("Synthetic Temperature Sensor")
plt.show()

### Splitting the Dataset

In [ ]:
# 70% of the dataset is used for training, the remaining 30% is split equally between validation and test sets.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

### Input Normalization

In [ ]:
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)
print("Mean:", normalizer.mean.numpy())
print("Variance:", normalizer.variance.numpy())

### Building the Regression Network

In [ ]:
regression_model = keras.Sequential([
    keras.Input(shape=(1,)),
    normalizer,
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])

In [ ]:
regression_model.summary()

In [ ]:
regression_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)

### Training the Regression model

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

In [ ]:
history = regression_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0
)

In [ ]:
print(
    "Training stopped after",
    len(history.history["loss"]),
    "epochs"
)

### Visualizing the training process

In [ ]:
plt.plot(
    history.history["loss"],
    label="Training loss"
)
plt.plot(
    history.history["val_loss"],
    label="Validation loss"
)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("Regression Model Training")
plt.legend()
plt.show()


### Evaluating the Regression Model

In [ ]:
test_mse, test_mae = regression_model.evaluate(
    X_test,
    y_test,
    verbose=0
)
print("Test MSE:", test_mse)
print("Test MAE:", test_mae, "°C")


In [ ]:
predictions = regression_model.predict(
    X_test,
    verbose=0
).flatten()
true_values = y_test.flatten()

In [ ]:
plt.scatter(
    true_values,
    predictions,
    alpha=0.5,
    color="tab:blue",
    label="Model predictions"
)

minimum = min(true_values.min(), predictions.min())
maximum = max(true_values.max(), predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    "--",
    color="tab:orange",
    label="Perfect prediction (y = x)"
)

plt.xlabel("True temperature [°C]")
plt.ylabel("Predicted temperature [°C]")
plt.title("True vs. Predicted Temperature")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Classification Problem

### Physionet 2016 Dataset manipulation

In [ ]:
physionet_2016_source_directory = r"<path to physionet cinc 2016 dataset>"
normal_recordings_destination_directory = r'<path to destination directory for normal recordings>'
murmur_recordings_destination_directory = r'<path to destination directory for murmur recordings>'
# Make sure the destination directories exist
os.makedirs(normal_recordings_destination_directory, exist_ok=True)
os.makedirs(murmur_recordings_destination_directory, exist_ok=True)
# TO BE RUN ONLY ONCE. TO BE COMMENTED OUT AFTERWARDS. Otherwise, it will overwrite the files in the destination directory every time.
relevant_subdirs = ["training-a", "training-b", "training-c", "training-d", "training-e", "training-f"]
for sub_dir in relevant_subdirs:
    sub_dir_path = os.path.join(physionet_2016_source_directory, sub_dir)
    reference_csv_path = os.path.join(sub_dir_path, "REFERENCE.csv")
    if os.path.isfile(reference_csv_path):
        data = pd.read_csv(reference_csv_path, header=None, names=["Recording ID", "Murmur Status"])
        for index, row in data.iterrows():
        # Use strip() for removing any leading or trailing white space
            recording_id = str(row["Recording ID"]).strip()
            murmur_status = int(row["Murmur Status"])  # -1 for normal, 1 for murmur
            recording_file = f"{recording_id}.wav"
            recording_file_source_path = os.path.join(sub_dir_path, recording_file)
            if not os.path.isfile(recording_file_source_path):
                print(f"Warning: File {recording_file} not found in {sub_dir_path}. Skipping.")
                continue
            if murmur_status == -1:
                new_file_name = f"{recording_id}_normal.wav"
                recording_file_destination_path = os.path.join(normal_recordings_destination_directory, new_file_name)
            elif murmur_status == 1:
                new_file_name = f"{recording_id}_murmur.wav"
                recording_file_destination_path = os.path.join(murmur_recordings_destination_directory, new_file_name)
            else:
                continue
            # Copy and rename the file
            shutil.copy2(recording_file_source_path, recording_file_destination_path)
            print(f"Copied and renamed: {recording_file} -> {recording_file_destination_path}")
print("Processing completed.")

### Physionet 2022 Dataset manipulation

In [ ]:
physionet_2022_source_directory = r'<path to the training_data folder of the PhysioNet2022 Challenge Dataset>'
physionet_2022_csv_path = r'<path to the training_data.csv file of the PhysioNet2022 Challenge Dataset>'
# TO BE RUN ONLY ONCE. TO BE COMMENTED OUT AFTERWARDS. Otherwise, it will overwrite the files in the destination directory every time.
physionet_2022_data = pd.read_csv(physionet_2022_csv_path)
for index, row in physionet_2022_data.iterrows():
    patient_id = str(row['Patient ID']).strip()
    status = str(row['Murmur']).strip()
    murmur_locations = str(row['Murmur locations']).strip() if status == "Present" else ""
    # e.g. : Murmur location = AV+MV+PV+TV
    murmur_locations_set = set(murmur_locations.split('+')) if murmur_locations else set()
    # Get all files for the patient
    patient_files = [f for f in os.listdir(physionet_2022_source_directory) if f.startswith(patient_id + "_") and f.endswith(".wav")]
    for file in patient_files:
        auscultation_location = file.split('_')[1].replace(".wav", "")
        # Ignore the patients with "Unknown" "Murmur" column status. They are considered inconclusive
        if status == "Absent":
            # splitext() returns a tuple (root, ext), so we take the first element [0] to get the root without extension
            new_file_name = f"{os.path.splitext(file)[0]}_normal.wav"
            destination_path = os.path.join(normal_recordings_destination_directory, new_file_name)
        elif status == "Present" and auscultation_location in murmur_locations_set:
            new_file_name = f"{os.path.splitext(file)[0]}_murmur.wav"
            destination_path = os.path.join(murmur_recordings_destination_directory, new_file_name)
        else:
            # Normal recording of a patient with murmurs
            new_file_name = f"{os.path.splitext(file)[0]}_normal.wav"
            destination_path = os.path.join(normal_recordings_destination_directory, new_file_name)
        # Source file path
        source_path = os.path.join(physionet_2022_source_directory, file)
        # Copy and rename the file
        shutil.copy2(source_path, destination_path)
        print(f"Copied and renamed: {file} -> {destination_path}")
print("Processing completed.")

### COUNT THE NUMBER OF .wav FILES FOR EACH CLASS :

In [ ]:
def count_wav_files_in_folder(folder_path, image_extensions=(".wav")):
    count = 0
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(image_extensions):
            count += 1
    return count
normal_recordings_count = count_wav_files_in_folder(normal_recordings_destination_directory)
murmur_recordings_count = count_wav_files_in_folder(murmur_recordings_destination_directory)
print(f"There are {normal_recordings_count} normal recordings in the merged dataset")
print(f"There are {murmur_recordings_count} recordings containing murmurs in the merged dataset")

In [ ]:
print(f"There are {normal_recordings_count + murmur_recordings_count} recordings in the merged dataset")

### COUNT THE PERCENTAGE OF EACH CLASS :

In [ ]:
total_recordings_count = normal_recordings_count + murmur_recordings_count
normal_percentage = (normal_recordings_count / total_recordings_count) * 100
murmur_percentage = (murmur_recordings_count / total_recordings_count) * 100
print(f"Normal recordings: {normal_recordings_count} ({normal_percentage:.2f}%)")
print(f"Murmur recordings: {murmur_recordings_count} ({murmur_percentage:.2f}%)")

### Preprocessing the merged dataset

In [ ]:
filtered_normal_recordings_destination_directory = r'<path to destination directory for filtered normal recordings>'
filtered_murmur_recordings_destination_directory = r'<path to destination directory for filtered murmur recordings>'
# Make sure the destination directories exist
os.makedirs(filtered_normal_recordings_destination_directory, exist_ok=True)
os.makedirs(filtered_murmur_recordings_destination_directory, exist_ok=True)

# A Butterworth low pass filter was chosen because it behaves close to an ideal filter
def butter_low_pass(samples, sample_rate, cutoff_freq, order, outputType):
    # Normalize the cutoff frequency
    max_freq = sample_rate / 2
    cutoff_freq_norm = cutoff_freq / max_freq
    # Calculate the Butterworth filter coefficients
    b, a = signalpy.butter(order, cutoff_freq_norm, btype='low')
    # Apply the filter to the signal
    s = signalpy.lfilter(b, a, samples).astype(outputType)
    return s
for normal_recording in os.listdir(normal_recordings_destination_directory) :
    recording_path = os.path.join(normal_recordings_destination_directory , normal_recording)
    # The parameter sr=None ensures that the original sample rate of the audio file is preserved when loading it with librosa.
    samples , sample_rate = librosa.load(recording_path , sr=None)
    # Relevant frequency range = 20-500Hz => Cutoff frequency = 600Hz is a good choice. 
    # The order of the filter is set to 11, which does not take too much time to compute and is enough for our purposes.
    samples_filtered = butter_low_pass(samples, sample_rate, 600, 11, np.float32)
    filtered_recording_file_name = f"{os.path.splitext(normal_recording)[0]}_filtered.wav"
    filtered_recording_path = os.path.join(filtered_normal_recordings_destination_directory, filtered_recording_file_name)
    wavfile.write(filtered_recording_path, sample_rate, samples_filtered)
for murmur_recording in os.listdir(murmur_recordings_destination_directory) :
    recording_path = os.path.join(murmur_recordings_destination_directory, murmur_recording)
    samples, sample_rate = librosa.load(recording_path, sr=None)
    samples_filtered = butter_low_pass(samples, sample_rate, 600, 11, np.float32)
    filtered_recording_file_name = f"{os.path.splitext(murmur_recording)[0]}_filtered.wav"
    filtered_recording_path = os.path.join(filtered_murmur_recordings_destination_directory, filtered_recording_file_name)
    wavfile.write(filtered_recording_path, sample_rate, samples_filtered)

### Generate and save the spectrograms of the filtered recordings

In [ ]:
def save_spectrogram(filename, output_dir):
    y, sr = librosa.load(filename, sr=None)
    n_fft = 8192  # Increase for better frequency resolution
    hop_length = 512  # Decrease for better time resolution
    win_length = 8192  # Generally the same as n_fft
    # Using the Short Time Fourier Transform to generate the spectrogram
    D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
    S = np.abs(D)
    S_dB = librosa.amplitude_to_db(S, ref=np.max)
    plt.figure(figsize=(600, 400, "px"))
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='linear')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.tight_layout()
    plt.ylim(0, 1000) # Visible frequency range 0-1000Hz
    basename = os.path.basename(filename)
    base_filename = os.path.splitext(basename)[0]
    output_filename = os.path.join(output_dir, base_filename + "_spectrogram.png")
    plt.savefig(output_filename)
    plt.close()

In [ ]:
# Move the filtered recordings into batches of 300 files.
# Only loose WAV files are included, so rerunning this cell skips existing batches.
def split_filtered_recordings_into_batches(recordings_directory, batch_size=300):
    recordings = sorted(
        recording
        for recording in os.listdir(recordings_directory)
        if recording.lower().endswith('.wav')
        and os.path.isfile(os.path.join(recordings_directory, recording))
    )
    # Equivalent to :
    # recordings = []
    # for recording in os.listdir(recordings_directory):
    #     recording_path = os.path.join(recordings_directory, recording)
    #     if recording.lower().endswith('.wav') and os.path.isfile(recording_path):
    #         recordings.append(recording)
    # recordings.sort()
    batch_directories = [
        os.path.join(recordings_directory, f'Batch{batch_number}')
        for batch_number in range(1, (len(recordings) + batch_size - 1) // batch_size + 1)
    ]
    # Avoid mixing new recordings into an existing batch or overwriting files.
    for batch_directory in batch_directories:
        if os.path.exists(batch_directory):
            raise FileExistsError(f'Batch folder already exists: {batch_directory}')
    for start, batch_directory in zip(range(0, len(recordings), batch_size), batch_directories):
        os.makedirs(batch_directory)
        for recording in recordings[start:start + batch_size]:
            #os.rename() moves the files in this case
            os.rename(
                os.path.join(recordings_directory, recording),
                os.path.join(batch_directory, recording),
            )
split_filtered_recordings_into_batches(filtered_normal_recordings_destination_directory)
split_filtered_recordings_into_batches(filtered_murmur_recordings_destination_directory)

In [ ]:
normal_spectrograms_dest_dir = r'<path to Normal spectrograms destination directory>'
murmur_spectrograms_dest_dir = r'<path to Murmur spectrograms destination directory>'
os.makedirs(normal_spectrograms_dest_dir, exist_ok=True)
os.makedirs(murmur_spectrograms_dest_dir, exist_ok=True)

# Generate spectrograms for one batch at a time to limit memory usage.
# Replace <add batch folder name> in both paths for the selected class, then
# uncomment its loop. Comment the loop out again after generating that batch.
# for normal_recording in os.listdir(os.path.join(filtered_normal_recordings_destination_directory, '<add batch folder name>')):
#     recording_path = os.path.join(filtered_normal_recordings_destination_directory, '<add batch folder name>', normal_recording)
#     save_spectrogram(recording_path, normal_spectrograms_dest_dir)
# for murmur_recording in os.listdir(os.path.join(filtered_murmur_recordings_destination_directory, '<add batch folder name>')):
#     recording_path = os.path.join(filtered_murmur_recordings_destination_directory, '<add batch folder name>', murmur_recording)
#     save_spectrogram(recording_path, murmur_spectrograms_dest_dir)

### Creating the data pipeline for training the neural network

In [ ]:
test_normal_spectrograms_dest_dir = r'<path to Testing Normal spectrograms destination directory>'
test_murmur_spectrograms_dest_dir = r'<path to Testing Murmur spectrograms destination directory>'
train_val_normal_spectrograms_dest_dir = r'<path to Training/Validation Normal spectrograms destination directory>'
train_val_murmur_spectrograms_dest_dir = r'<path to Training/Validation Murmur spectrograms destination directory>'
# TO BE RUN ONLY ONCE. TO BE COMMENTED OUT AFTERWARDS
percentage_of_img_to_move = 0.15
normal_spectrograms = [f for f in os.listdir(train_val_normal_spectrograms_dest_dir) if f.endswith(".png")]
num_of_normal_spectrograms_to_move = int(len(normal_spectrograms)*percentage_of_img_to_move)
random.seed(42)  # For reproducibility
normal_spectrograms_to_move = random.sample(normal_spectrograms, num_of_normal_spectrograms_to_move)
for normal_spectrogam in normal_spectrograms_to_move:
    source_path = os.path.join(train_val_normal_spectrograms_dest_dir, normal_spectrogam)
    destination_path = os.path.join(test_normal_spectrograms_dest_dir, normal_spectrogam)
    shutil.move(source_path, destination_path)
    print(f"Moved: {normal_spectrogam} -> {destination_path}")
murmur_spectrograms = [f for f in os.listdir(train_val_murmur_spectrograms_dest_dir) if f.endswith(".png")]
num_of_murmur_spectrograms_to_move = int(len(murmur_spectrograms)*percentage_of_img_to_move)
random.seed(42)  # For reproducibility
murmur_spectrograms_to_move = random.sample(murmur_spectrograms, num_of_murmur_spectrograms_to_move)
for murmur_spectrogam in murmur_spectrograms_to_move:
    source_path = os.path.join(train_val_murmur_spectrograms_dest_dir, murmur_spectrogam)
    destination_path = os.path.join(test_murmur_spectrograms_dest_dir, murmur_spectrogam)
    shutil.move(source_path, destination_path)
    print(f"Moved: {murmur_spectrogam} -> {destination_path}")

In [ ]:
train_val_spectrograms_dest_dir = r'<path to the training/validation folder – parent folder of the Normal/Murmur folders>'
heart_sound_recordings_spectrograms_dataset = tf.keras.utils.image_dataset_from_directory(
    train_val_spectrograms_dest_dir,
    labels='inferred',
    label_mode='binary',  # For binary classification
    batch_size=32,
    image_size=(400, 600),  # The chosen size of the spectrograms images
    shuffle=True,
    seed=123
)
class_names = heart_sound_recordings_spectrograms_dataset.class_names
print("Class mapping:", {i: class_names[i] for i in range(len(class_names))})
for images, labels in heart_sound_recordings_spectrograms_dataset.take(1):  # Take one batch only
    plt.figure(figsize=(12, 12))
    for i in range(9):  # Show the first 9 images
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))  # Convert tensor to image format
        plt.title(f"Label: {class_names[labels[i].numpy().argmax()]}")  # Display the class index
        plt.axis("off")
    plt.show()
training_dataset_size = int(0.875*len(heart_sound_recordings_spectrograms_dataset))
validation_dataset_size = int(0.125*len(heart_sound_recordings_spectrograms_dataset))
training_dataset = heart_sound_recordings_spectrograms_dataset.take(training_dataset_size)
validation_dataset = heart_sound_recordings_spectrograms_dataset.skip(training_dataset_size).take(validation_dataset_size)
# Optimize datasets for performance
training_dataset = training_dataset.cache()
validation_dataset = validation_dataset.cache()
training_dataset = training_dataset.prefetch(tf.data.AUTOTUNE)
validation_dataset = validation_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
test_spectrograms_dest_dir = r'<path to Testing Spectrograms Dataset directory>'
testing_dataset = tf.keras.utils.image_dataset_from_directory(
    test_spectrograms_dest_dir,
    labels='inferred',
    label_mode='binary',  # For binary classification
    batch_size=8,
    image_size=(400, 600),
    shuffle=False,
)

testing_dataset = testing_dataset.cache()
testing_dataset = testing_dataset.prefetch(tf.data.AUTOTUNE)

### Training the CNN

In [ ]:
input = tf.keras.Input(shape=(400,600,3))
x = tf.keras.layers.Conv2D(96, (3,3), activation='relu')(input)
x = tf.keras.layers.MaxPooling2D((2, 2))(x)
x = tf.keras.layers.Conv2D(32, (5,5), activation='relu')(x)
x = tf.keras.layers.MaxPooling2D((2, 2))(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.4)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)
CNN_model = tf.keras.Model(inputs=input, outputs=output)
CNN_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Recall()])

In [ ]:
CNN_model.summary()

In [ ]:
CNN_model_history = CNN_model.fit(training_dataset, epochs=10, validation_data = validation_dataset)

In [ ]:
CNN_model.save("CNN_model.keras")

In [ ]:
def plot_training_history(history):
    epochs = range(1, len(history.history['accuracy']) + 1)
    # Plot Accuracy
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history['accuracy'], 'b', label='Training Accuracy')
    plt.plot(epochs, history.history['val_accuracy'], 'r', label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    # Plot Recall
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history.history['recall'], 'b', label='Training Recall')
    plt.plot(epochs, history.history['val_recall'], 'r', label='Validation Recall')
    plt.title('Training and Validation Recall')
    plt.xlabel('Epochs')
    plt.ylabel('Recall')
    plt.legend()
    plt.show()
plot_training_history(CNN_model_history)

### Evaluate the CNN on the testing dataset

In [ ]:
test_loss, test_accuracy, test_recall = CNN_model.evaluate(testing_dataset)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test recall: {test_recall:.4f}")

In [ ]:
# testing_dataset = testing_dataset.unbatch() #Comment out if running this cell multiple times
y_true, y_pred = [], []
for images, labels in testing_dataset:
    predictions = CNN_model.predict(tf.expand_dims(images, axis=0))
    predicted_class = predictions[0][0]
    true_class = labels.numpy()
    y_pred.append(predicted_class)
    y_true.append(true_class)
y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
y_pred_copy = y_pred
y_pred_copy = [1 if prediction > 0.54 else 0 for prediction in y_pred_copy]

In [ ]:
print(y_pred_copy)

In [ ]:
print(y_pred)

In [ ]:
conf_matrix = confusion_matrix(y_true, y_pred_copy)
# Display the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()
plt.savefig("CNN_confusion_matrix.png", dpi=300)
# Classification report
classification_report_str = classification_report(y_true, y_pred_copy, target_names=class_names)
print("Classification Report:\n", classification_report_str)
with open("CNN_classification_report.txt", "w") as file:
    file.write(classification_report_str)